# Create anndata for peakVI

In [7]:
here::i_am("rna/trajectories/infer_trajectory.R")

suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(scater))


# Load default settings
source(here::here("settings.R"))
source(here::here("utils.R"))

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/code



In [8]:
args <- list()
args$sce <-file.path(io$basedir,"data/processed/atac/archR/Matrices/PeakMatrix_summarized_experiment.rds")
args$metadataRNA <- file.path(io$basedir,"results/rna/mapping/sample_metadata_after_mapping.txt.gz")
args$metadataATAC <- file.path(io$basedir, '/results/atac/archR/qc/sample_metadata_after_qc.txt.gz')
args$trajectory_name <- "epiblast_blood"
args$celltype_label <- "celltype"
args$outdir <- file.path(io$basedir,"results/epiblast_blood")

args$batch_variable = 'sample'
args$features = 3000
args$n_pcs <- 25
args$sample <- c('E7.5_rep1','E7.5_rep2','E7.75_rep1','E8.0_rep1','E8.0_rep2','E8.5_rep1','E8.5_rep2','E8.75_rep1','E8.75_rep2')
## END TEST ##

# I/O
dir.create(args$outdir, showWarnings=F, recursive=T)

# Trajectory 
opts$celltypes = c("Epiblast",
                    "Primitive_Streak" ,
                    "Nascent_mesoderm",
                    "ExE_mesoderm",
                    "Mixed_mesoderm",
                    "Allantois",
                    "Mesenchyme",
                    "Haematoendothelial_progenitors",
                    "Endothelium",
                    "Blood_progenitors_1",
                    "Blood_progenitors_2",
                    "Erythroid1",
                    "Erythroid2",
                    "Erythroid3")

In [10]:
##########################
## Load sample metadata ##
##########################

sample_metadata <- fread(args$metadataRNA) %>%
  .[pass_rnaQC==TRUE & doublet_call==FALSE] %>%
    merge(.,fread(args$metadataATAC)[,c('cell', 'pass_atacQC')], by='cell') %>%
    .[pass_atacQC==TRUE]

stopifnot(args$celltype_label%in%colnames(sample_metadata))
sample_metadata <- sample_metadata   %>%
  .[,celltype:=eval(as.name(args$celltype_label))] %>%
  .[celltype%in%opts$celltypes] %>%
  .[,celltype:=factor(celltype,levels=opts$celltypes)] %>%
  .[sample%in% args$sample]

table(sample_metadata$celltype)


                      Epiblast               Primitive_Streak 
                           984                            542 
              Nascent_mesoderm                   ExE_mesoderm 
                          1327                           1173 
                Mixed_mesoderm                      Allantois 
                           265                            653 
                    Mesenchyme Haematoendothelial_progenitors 
                          3123                            913 
                   Endothelium            Blood_progenitors_1 
                           666                            230 
           Blood_progenitors_2                     Erythroid1 
                           565                           1119 
                    Erythroid2                     Erythroid3 
                           939                            187 

In [11]:
#############################
## Load ATAC accessibility ##
#############################
  
sce = readRDS(args$sce)
sce = sce[,sample_metadata$cell]
sce = as(sce, 'SingleCellExperiment')

# Filter regions accessibile in 5% of cells
nCells = 0.05 * nrow(sample_metadata)
region_counts = rowSums(assay(sce, 'PeakMatrix')>0)
keep_regions = region_counts[region_counts>nCells]
sce = sce[names(keep_regions), ]

assay(sce, 'counts') = assay(sce, 'PeakMatrix')
assay(sce, 'PeakMatrix') = NULL

colData(sce) <- sample_metadata %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce),] %>% DataFrame()

In [9]:
sceasy::convertFormat(sce, from="sce", 
                      to="anndata",
                      outFile= sprintf("%s/anndata_ATAC.h5ad",args$outdir))

Warning message in .regularise_df(as.data.frame(SummarizedExperiment::colData(obj)), :
“Dropping single category variables:genotype, pass_rnaQC, doublet_call, pass_atacQC”


AnnData object with n_obs × n_vars = 12686 × 76538
    obs: 'barcode', 'sample', 'nFeature_RNA', 'nCount_RNA', 'mitochondrial_percent_RNA', 'ribosomal_percent_RNA', 'stage', 'doublet_score', 'celltype', 'celltype.score', 'closest.cell'
    var: 'idx'

In [13]:
args$outdir = file.path(io$basedir, 'results/epiblast_blood/atac/')
features = fread(sprintf("%s/atac_variable_features.txt.gz",args$outdir))

In [14]:
head(features)

seqnames,idx,start,end,rowSums,feature
<chr>,<int>,<int>,<int>,<dbl>,<chr>
chr1,7,3482876,3483476,5916,chr1:3482876-3483476
chr1,13,3670515,3671115,35116,chr1:3670515-3671115
chr1,14,3671511,3672111,50843,chr1:3671511-3672111
chr1,41,4491836,4492436,28676,chr1:4491836-4492436
chr1,42,4492444,4493044,20847,chr1:4492444-4493044
chr1,43,4493400,4494000,33319,chr1:4493400-4494000


In [17]:
sce = readRDS(args$sce)
sce = sce[,sample_metadata$cell]
sce = as(sce, 'SingleCellExperiment')

sce = sce[features$feature,]

assay(sce, 'counts') = assay(sce, 'PeakMatrix')
assay(sce, 'PeakMatrix') = NULL

colData(sce) <- sample_metadata %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce),] %>% DataFrame()

In [19]:
# Anndata with only top 45k variable features 
args$outdir = file.path(io$basedir, 'results/epiblast_blood/atac/')
sceasy::convertFormat(sce, from="sce", 
                      to="anndata",
                      outFile= sprintf("%s/anndata_ATAC_variable.h5ad",args$outdir))

Warning message in .regularise_df(as.data.frame(SummarizedExperiment::colData(obj)), :
“Dropping single category variables:genotype, pass_rnaQC, doublet_call, pass_atacQC”


AnnData object with n_obs × n_vars = 12686 × 30000
    obs: 'barcode', 'sample', 'nFeature_RNA', 'nCount_RNA', 'mitochondrial_percent_RNA', 'ribosomal_percent_RNA', 'stage', 'doublet_score', 'celltype', 'celltype.score', 'closest.cell'
    var: 'idx'